<a href="https://colab.research.google.com/github/rasia92015-byte/NHANES/blob/main/Notebooks/BChe_PFAS/PFAS_BChE_NHANES.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# NHANES 2021–2023 PFAS–BChE:

Upload xpt files into /content:
DEMO_L.xpt, PFAS_L.xpt, BCHE_L.xpt, BMX_L.xpt, SMQ_L.xpt, ALQ_L.xpt, PAQ_L.xpt.

≥20 years; four individual log₂(PFAS) exposures; ln(BChE activity)/ln(unadducted concentration); age, sex, race/ethnicity, BMI, smoking 3 levels, drinking in past year 0/1, any moderate/vigorous leisure activity 0/1; PFAS-specific WTSPF2YR; SDMVSTRA/SDMVPSU; full-design zero-score domain Taylor linearization, df = 15; t tests and 95% CIs. Interaction models include PFAS × female and are exploratory.


## 1. Settings and packages

In [ ]:
"""from raw NHANES 2021-2023 XPT.

"""

import os
from pathlib import Path
import json
import platform
import warnings

import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt

INPUT = Path(os.environ.get("PFAS_INPUT_DIR", "/content"))
OUTPUT = Path(os.environ.get("PFAS_OUTPUT_DIR", str(INPUT / "PFAS_BChE_output")))
OUTPUT.mkdir(parents=True, exist_ok=True)
REQUIRED = ["DEMO_L", "PFAS_L", "BCHE_L", "BMX_L", "SMQ_L", "ALQ_L", "PAQ_L"]
OPTIONAL = ["BIOPRO_L"]
PFAS = {
    "n-PFOA": ("LBXNFOA", "LBDNFOAL"),
    "n-PFOS": ("LBXNFOS", "LBDNFOSL"),
    "PFNA": ("LBXPFNA", "LBDPFNAL"),
    "PFHxS": ("LBXPFHS", "LBDPFHSL"),
}
COVARIATES = ["age", "sex", "race", "bmi", "smoking", "alcohol", "activity"]
CATEGORICAL = ("sex", "race", "smoking", "alcohol", "activity")
SURVEY = ["WTSPF2YR", "SDMVSTRA", "SDMVPSU"]

# check values
EXPECTED = {
    ("n-PFOA", "activity"): (1.72, 0.67, 2.79, 0.0032, 1844),
    ("n-PFOA", "concentration"): (2.07, 0.68, 3.48, 0.0062, 1843),
    ("n-PFOS", "activity"): (1.60, 0.32, 2.90, 0.0177, 1844),
    ("n-PFOS", "concentration"): (1.66, 0.36, 2.98, 0.0159, 1843),
    ("PFNA", "activity"): (2.38, 1.07, 3.70, 0.0014, 1844),
    ("PFNA", "concentration"): (2.00, 0.36, 3.67, 0.0199, 1843),
    ("PFHxS", "activity"): (1.14, -0.26, 2.55, 0.1028, 1844),
    ("PFHxS", "concentration"): (1.71, 0.34, 3.10, 0.0178, 1843),
}
EXPECTED_INTERACTION_P = {
    ("n-PFOA", "activity"): 0.1051, ("n-PFOA", "concentration"): 0.0966,
    ("n-PFOS", "activity"): 0.5217, ("n-PFOS", "concentration"): 0.5694,
    ("PFNA", "activity"): 0.0111, ("PFNA", "concentration"): 0.0154,
    ("PFHxS", "activity"): 0.3697, ("PFHxS", "concentration"): 0.4032,
}




## 2. Read raw XPT, merge by SEQN and code covariates

In [ ]:
def require_inputs():
    absent = [f"{s}.xpt" for s in REQUIRED if not (INPUT / f"{s}.xpt").is_file()]
    if absent:
        raise FileNotFoundError(
            f"Missing XPT file(s) in {INPUT}: {', '.join(absent)}. "
            "Place the raw XPT files directly in /content (not in a zip or subfolder)."
        )


def read_merge():
    require_inputs()
    names = REQUIRED + [s for s in OPTIONAL if (INPUT / f"{s}.xpt").is_file()]
    parts = {}
    for name in names:
        frame = pd.read_sas(INPUT / f"{name}.xpt", format="xport")
        if "SEQN" not in frame.columns or frame.SEQN.isna().any() or not frame.SEQN.is_unique:
            raise ValueError(f"Invalid/duplicate SEQN in {name}.xpt")
        parts[name] = frame
        print(f"Read {name}.xpt: {len(frame):,} participants, {len(frame.columns)} columns")
    if "BIOPRO_L" not in parts:
        print("BIOPRO_L.xpt not found (optional; not used in the manuscript's three tables).")
    data = parts["DEMO_L"].copy()
    for name in names:
        if name == "DEMO_L":
            continue
        data = data.merge(parts[name], on="SEQN", how="left", validate="one_to_one", suffixes=("", f"_{name}"))
    # SAS XPORT's representation of numeric zero may appear as subnormal positive values.
    # Here, 3,618 PFAS_L rows include 1,067 WTSPF2YR values equal to ~5.397605e-79.

    for col in data.select_dtypes(include="number").columns:
        data.loc[(data[col] > 0) & (data[col] < 1e-40), col] = 0.0
    checkcols = ["SEQN", "RIDAGEYR", "RIAGENDR", "RIDRETH3", "BMXBMI", "SMQ020", "SMQ040",
                 "ALQ111", "ALQ121", "PAD790Q", "PAD810Q", "LBXBA", "LBXCH"]
    checkcols += SURVEY + [v for x in PFAS.values() for v in x]
    missing_cols = sorted(set(checkcols) - set(data.columns))
    if missing_cols:
        raise ValueError(f"Expected XPT variables not found: {missing_cols}")
    # Full positive-subsample-weight design includes people outside age>=20/complete-case domains.
    # Do NOT take XPT SAS tiny zero values as nonzero: it would incorrectly inflate this count.
    full = data.loc[(data.WTSPF2YR > 0) & data.SDMVSTRA.notna() & data.SDMVPSU.notna()].copy()
    full_psus = full.groupby(["SDMVSTRA", "SDMVPSU"], sort=True).size().index
    n_strata = full.SDMVSTRA.nunique()
    design_df = len(full_psus) - n_strata
    if any(g < 2 for g in full.groupby("SDMVSTRA").SDMVPSU.nunique()):
        raise ValueError("Full survey design has a stratum with fewer than 2 PSUs")
    print(f"Complete positive-weight PFAS design: {len(full):,} persons, {n_strata} strata, "
          f"{len(full_psus)} PSUs, df = {design_df}")

    data["age"] = data.RIDAGEYR
    data["sex"] = data.RIAGENDR
    data["race"] = data.RIDRETH3
    data["bmi"] = data.BMXBMI
    # Never <100 lifetime; former >=100 but not currently; current >=100 and now every/some days.
    data["smoking"] = np.select([
        data.SMQ020.eq(2),
        data.SMQ020.eq(1) & data.SMQ040.eq(3),
        data.SMQ020.eq(1) & data.SMQ040.isin([1, 2])],
        [0, 1, 2], default=np.nan)
    # Historic locked binary alcohol coding: any drinking in prior 12 months.
    data["alcohol"] = np.select([
        data.ALQ111.eq(2) | data.ALQ121.eq(0),
        data.ALQ121.between(1, 10)], [0, 1], default=np.nan)
    # Historic locked binary leisure activity coding. Unknowns remain missing.
    mod = np.where(data.PAD790Q.eq(0), 0,
                   np.where(data.PAD790Q.between(1, 365), 1, np.nan))
    vig = np.where(data.PAD810Q.eq(0), 0,
                   np.where(data.PAD810Q.between(1, 365), 1, np.nan))
    data["activity"] = np.where((mod == 1) | (vig == 1), 1,
                                np.where((mod == 0) & (vig == 0), 0, np.nan))
    adults = data.loc[(data.RIDAGEYR >= 20) & (data.WTSPF2YR > 0) &
                      data.SDMVSTRA.notna() & data.SDMVPSU.notna()].copy()
    print(f"Adult positive-weight analytic frame: {len(adults):,}")
    return adults, full_psus, design_df, len(full), n_strata




## 3. Survey-weighted domain Taylor covariance and effect contrasts

In [ ]:
def model_sample(adults, pf, target):
    var, lod_flag = PFAS[pf]
    # NHANES concentrations below LLOD already have substituted concentrations in LBX... columns.
    # LBD...L: 0 above LLOD; 1 below LLOD; reject missing/invalid indicators.
    good = (adults[var] > 0) & adults[lod_flag].isin([0, 1]) & (adults[target] > 0)
    z = adults.loc[good].copy()
    z["logpfas"] = np.log2(z[var])
    z["lnoutcome"] = np.log(z[target])
    return z.dropna(subset=COVARIATES + SURVEY + ["logpfas", "lnoutcome"])


def survey_domain_fit(z, y_col, terms, full_psus, design_df, categories=CATEGORICAL):
    """Weighted estimating equations + design-based Taylor covariance with zero domain PSU scores.

    Full-domain PSU totals include zeros for survey PSUs having no eligible model cases.
    The covariance is A^-1 * sum_h[m_h/(m_h-1) * centered_PSU_scores'*centered_PSU_scores] * A^-1.
    """
    z = z.dropna(subset=[y_col] + list(terms) + SURVEY).copy()
    x = z[list(terms)].copy()
    for cat in categories:
        if cat in x.columns:
            x[cat] = x[cat].astype("category")
    x = pd.get_dummies(x, drop_first=True, dtype=float)
    x.insert(0, "intercept", 1.0)
    X = x.to_numpy(float)
    Y = z[y_col].to_numpy(float)
    W = z.WTSPF2YR.to_numpy(float)
    if np.linalg.matrix_rank(X) != X.shape[1]:
        raise ValueError(f"Model design matrix rank deficient: {list(x.columns)}")
    bread = np.linalg.inv(X.T @ (W[:, None] * X))
    beta = bread @ (X.T @ (W * Y))
    score = (W * (Y - X @ beta))[:, None] * X
    tmp = pd.DataFrame(score, columns=x.columns)
    tmp["SDMVSTRA"] = z.SDMVSTRA.to_numpy()
    tmp["SDMVPSU"] = z.SDMVPSU.to_numpy()
    psus = (tmp.groupby(["SDMVSTRA", "SDMVPSU"], sort=True).sum()
            .reindex(full_psus, fill_value=0.0))
    middle = np.zeros((len(beta), len(beta)))
    for _, group in psus.groupby(level=0, sort=True):
        cluster_scores = group.to_numpy(dtype=float)
        m = len(cluster_scores)
        if m <= 1:
            raise ValueError("Lonely PSU in full design")
        centered = cluster_scores - cluster_scores.mean(axis=0)
        middle += (m / (m - 1)) * (centered.T @ centered)
    covariance = bread @ middle @ bread
    return {"beta": pd.Series(beta, index=x.columns),
            "cov": pd.DataFrame(covariance, index=x.columns, columns=x.columns),
            "N": len(z), "df": design_df,
            "inside_PSU": len(tmp.groupby(["SDMVSTRA", "SDMVPSU"]))}


def contrast(model, terms):
    """Estimate linear contrast and its 95% t CI; raw P without FDR."""
    weights = pd.Series(terms, dtype=float)
    names = list(weights.index)
    b = model["beta"].loc[names].to_numpy()
    v = model["cov"].loc[names, names].to_numpy()
    vec = weights.to_numpy()
    estimate = float(vec @ b)
    se = float(np.sqrt(vec @ v @ vec))
    crit = stats.t.ppf(0.975, model["df"])
    p = float(2 * stats.t.sf(abs(estimate / se), model["df"]))
    return {"beta": estimate, "se": se, "pct": 100 * np.expm1(estimate),
            "lo": 100 * np.expm1(estimate - crit * se),
            "hi": 100 * np.expm1(estimate + crit * se), "p": p,
            "N": model["N"], "df": model["df"]}


def weighted_mean(data, values):
    return float(np.average(values, weights=data.WTSPF2YR.to_numpy(float)))




## 4. weighted baseline and PFAS detection

In [ ]:
def build_table1(adults):

    z = model_sample(adults, "n-PFOA", "LBXBA")
    for pf, (x, flag) in PFAS.items():
        z = z.loc[(z[x] > 0) & z[flag].isin([0, 1])]
    if len(z) == 0:
        raise ValueError("No Table 1 participants")
    data = [
        ("Age, years (mean)", weighted_mean(z, z.age)),
        ("Women, %", weighted_mean(z, 100 * z.sex.eq(2).astype(float))),
        ("Body mass index, kg/m² (mean)", weighted_mean(z, z.bmi)),
        ("Current smoking, %", weighted_mean(z, 100 * z.smoking.eq(2).astype(float))),
        ("Any past-year alcohol use, %", weighted_mean(z, 100 * z.alcohol.eq(1).astype(float))),
        ("Any moderate/vigorous leisure activity, %", weighted_mean(z, 100 * z.activity.eq(1).astype(float))),
    ]
    races = [(1, "Mexican American, %"), (2, "Other Hispanic, %"),
             (3, "Non-Hispanic White, %"), (4, "Non-Hispanic Black, %"),
             (6, "Non-Hispanic Asian, %"), (7, "Other/multiracial, %")]
    data += [(name, weighted_mean(z, 100 * z.race.eq(code).astype(float))) for code, name in races]
    for pf, (var, flag) in PFAS.items():
        sub = adults.loc[(adults[var] > 0) & adults[flag].isin([0, 1])]
        data.append((f"{pf} detected, %", weighted_mean(sub, 100 * sub[flag].eq(0).astype(float))))
    table = pd.DataFrame(data, columns=["Characteristic", "Weighted estimate"])
    table["Weighted estimate"] = table["Weighted estimate"].round(1)
    table.to_csv(OUTPUT / "Table1_Weighted_Characteristics.csv", index=False)
    return table, len(z)




## 5. primary and sex-interaction models

In [ ]:
def build_models(adults, full_psus, design_df):
    prim, inter = [], []
    for pf in PFAS:
        for target, label in [("LBXBA", "activity"), ("LBXCH", "concentration")]:
            z = model_sample(adults, pf, target)
            main_model = survey_domain_fit(z, "lnoutcome", ["logpfas"] + COVARIATES,
                                           full_psus, design_df)
            main = contrast(main_model, {"logpfas": 1.0})
            prim.append({"PFAS": pf, "outcome": label, **main})
            z["female_indicator"] = z.sex.eq(2).astype(float)
            z["pfas_female"] = z.logpfas * z.female_indicator
            sex_model = survey_domain_fit(
                z, "lnoutcome", ["logpfas", "female_indicator", "pfas_female"] +
                [t for t in COVARIATES if t != "sex"], full_psus, design_df,
                categories=("race", "smoking", "alcohol", "activity"))
            male = contrast(sex_model, {"logpfas": 1.0})
            female = contrast(sex_model, {"logpfas": 1.0, "pfas_female": 1.0})
            interaction = contrast(sex_model, {"pfas_female": 1.0})
            inter.append({"PFAS": pf, "outcome": label, "N": sex_model["N"],
                          "design_df": design_df, "male_pct": male["pct"],
                          "male_lo": male["lo"], "male_hi": male["hi"],
                          "female_pct": female["pct"], "female_lo": female["lo"],
                          "female_hi": female["hi"], "p_interaction": interaction["p"]})
    return pd.DataFrame(prim), pd.DataFrame(inter)




## 6. Figure 1

In [ ]:
def save_tables(prim, inter):
    prim.to_csv(OUTPUT / "Table2_Numeric.csv", index=False)
    inter.to_csv(OUTPUT / "Table3_Numeric.csv", index=False)
    def maincell(row):
        return f"{row.pct:+.2f} ({row.lo:.2f}, {row.hi:.2f}); P = {row.p:.4f}"
    wide = []
    for pf in PFAS:
        a = prim.loc[(prim.PFAS == pf) & (prim.outcome == "activity")].iloc[0]
        c = prim.loc[(prim.PFAS == pf) & (prim.outcome == "concentration")].iloc[0]
        wide.append({"PFAS": pf, "BChE activity, % (95% CI); P": maincell(a),
                     "Unadducted concentration, % (95% CI); P": maincell(c)})
    t2 = pd.DataFrame(wide)
    t2.to_csv(OUTPUT / "Table2_Primary_Associations.csv", index=False)
    sex_rows = []
    for r in inter.itertuples(index=False):
        label = "Activity" if r.outcome == "activity" else "Concentration"
        sex_rows.append({"PFAS / outcome": f"{r.PFAS} / {label}",
                         "Men: % difference (95% CI)": f"{r.male_pct:.2f}% ({r.male_lo:.2f}% to {r.male_hi:.2f}%)",
                         "Women: % difference (95% CI)": f"{r.female_pct:.2f}% ({r.female_lo:.2f}% to {r.female_hi:.2f}%)",
                         "P for interaction": f"{r.p_interaction:.4f}"})
    t3 = pd.DataFrame(sex_rows)
    t3.to_csv(OUTPUT / "Table3_Sex_Interactions.csv", index=False)
    return t2, t3


def make_figure1(primary):

    fig, ax = plt.subplots(figsize=(10.0, 6.5))
    labels = []
    blues, oranges = "#1a6482", "#d29643"
    for i, pf in enumerate(PFAS):
        for j, (outcome, color) in enumerate([("activity", blues), ("concentration", oranges)]):
            r = primary[(primary.PFAS == pf) & (primary.outcome == outcome)].iloc[0]
            y = i * 2 + j
            labels.append(f"{pf} — {outcome}")
            ax.errorbar(r.pct, y,
                        xerr=np.array([[r.pct - r.lo], [r.hi - r.pct]]),
                        fmt="o", color=color, markersize=7, elinewidth=1.7,
                        capsize=4, capthick=1.5)
    ax.axvline(0, linestyle="--", color="#777777", linewidth=1.0)
    ax.set_yticks(np.arange(8), labels)
    ax.set_ylim(7.5, -0.5)
    ax.set_xlabel("Percentage difference per exposure doubling (95% CI)")
    ax.set_xlim(-1.4, 5.1)
    ax.set_xticks(np.arange(-1, 6, 1))
    ax.grid(axis="x", alpha=.18)
    ax.set_axisbelow(True)
    fig.tight_layout()
    fig.savefig(OUTPUT / "Figure1_PFAS_BChE.png", dpi=250, bbox_inches="tight")
    fig.savefig(OUTPUT / "Figure1_PFAS_BChE.svg", bbox_inches="tight")
    plt.close(fig)




## 8. Run everything

In [ ]:
main()

# Colab inline preview of all tables and the single requested figure.
from IPython.display import display, Image
for basename in ['Table1_Weighted_Characteristics.csv',
                 'Table2_Primary_Associations.csv',
                 'Table3_Sex_Interactions.csv']:
    print('\n' + basename)
    display(pd.read_csv(OUTPUT / basename))
display(Image(filename=str(OUTPUT / 'Figure1_PFAS_BChE.png'), width=900))

# A ZIP of the generated files (no original XPT files included).
from zipfile import ZipFile, ZIP_DEFLATED
zip_path = INPUT / 'PFAS_BChE_3Tables_1Figure_results.zip'
with ZipFile(zip_path, 'w', compression=ZIP_DEFLATED) as bundle:
    for result in sorted(OUTPUT.iterdir()):
        if result.is_file(): bundle.write(result, arcname=result.name)
print('Saved ZIP:', zip_path)

